In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# 1. Load Dataset
df = pd.read_csv("Lipstick.csv")
print("Original data:\n", df, "\n")

# 2. Encode Categorical Values — one encoder per column, with safe casting
label_encoders = {}
for col in df.columns:
    # encode only non-numeric (object / category) columns
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# Print mappings
print("Mappings (class -> numeric):")
for col, le in label_encoders.items():
    print(f" {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print()

# 3. Split into Features (X) and Target (y)
X = df[['Age','Income','Gender','Ms']]
y = df['Buys']

# 4. Train Decision Tree Model
model = DecisionTreeClassifier(criterion="entropy", random_state=42)
model.fit(X, y)

# 5. Test Data (Given in Question for Practical 6)
# Age > 35, Income = Medium, Gender = Female, Ms = Married
test_raw = {'Age': '>35', 'Income': 'Medium', 'Gender': 'Female', 'Ms': 'Married'}

# Encode the test case using same label encoders (transform, not fit)
encoded_test = []
for col in X.columns:
    le = label_encoders.get(col)
    if le is None:
        raise ValueError(f"No encoder found for column {col}")
    val = str(test_raw[col])
    if val not in le.classes_:
        raise ValueError(f"Unseen category '{val}' for column '{col}'. Allowed: {list(le.classes_)}")
    encoded_test.append(int(le.transform([val])[0]))

print("Encoded Test Input:", encoded_test)

# 6. Make Prediction
prediction = model.predict([encoded_test])[0]
pred_label = label_encoders['Buys'].inverse_transform([prediction])[0]

print("\nPrediction (0 = No, 1 = Yes):", prediction)
print("Final Decision:", pred_label)

Original data:
     Id    Age  Income  Gender       Ms Buys
0    1    <21    High    Male   Single   No
1    2    <21    High    Male  Married   No
2    3  21-35    High    Male   Single  Yes
3    4    >35  Medium    Male   Single  Yes
4    5    >35     Low  Female   Single  Yes
5    6    >35     Low  Female  Married   No
6    7  21-35     Low  Female  Married  Yes
7    8    <21  Medium    Male   Single   No
8    9    <21     Low  Female  Married  Yes
9   10    >35  Medium  Female   Single  Yes
10  11    <21  Medium  Female  Married  Yes
11  12  21-35  Medium    Male  Married  Yes
12  13  21-35    High  Female   Single  Yes
13  14    >35  Medium    Male  Married   No 

Mappings (class -> numeric):
 Age: {'21-35': np.int64(0), '<21': np.int64(1), '>35': np.int64(2)}
 Income: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
 Gender: {'Female': np.int64(0), 'Male': np.int64(1)}
 Ms: {'Married': np.int64(0), 'Single': np.int64(1)}
 Buys: {'No': np.int64(0), 'Yes': np.int64(

D:\Anaconda\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
